In [ ]:
"""
Pareto Flow Size Distribution
==============================
Distributes network flows across size tiers using the Pareto (power-law) distribution.
Tiers: 1KB, 10KB, 100KB, 1MB, 10MB, 100MB, 1GB
"""

import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as ticker

# ── Configuration ─────────────────────────────────────────────────────────────

ALPHA       = 0.5       # Pareto shape parameter (typical internet: 1.0–1.5)
TOTAL_FLOWS = 10_000    # Total number of flows to distribute
SEED        = 1        # Random seed for reproducibility

TIERS = [
    ("10 KB",  10 * 1024),
    ("100 KB", 100 * 1024),
    ("1 MB",   1024 ** 2),
    ("10 MB",  10 * 1024 ** 2),
    ("100 MB", 100 * 1024 ** 2),
    ("1 GB",   1024 ** 3),
    ("10 GB",  10 * 1024 ** 3),
]

# ── Helper functions ───────────────────────────────────────────────────────────

def fmt_bytes(b: float) -> str:
    """Human-readable byte size."""
    for unit, threshold in [("GB", 1024**3), ("MB", 1024**2), ("KB", 1024)]:
        if b >= threshold:
            return f"{b / threshold:.2f} {unit}"
    return f"{b:.0f} B"


def pareto_tier_weights(alpha: float, tiers: list) -> np.ndarray:
    """
    Compute the probability weight for each tier using the Pareto tail:
        P(X > x) = (x_min / x) ^ alpha

    Weight for tier i  =  P(X >= tier_i) - P(X >= tier_{i+1})
    Last tier absorbs the remaining tail probability.
    """
    x_min = tiers[0][1]                          # minimum flow size (scale)
    tail  = lambda x: (x_min / x) ** alpha       # Pareto CCDF

    weights = []
    for i, (_, size) in enumerate(tiers):
        upper = tail(tiers[i + 1][1]) if i + 1 < len(tiers) else 0.0
        weights.append(tail(size) - upper)

    weights = np.array(weights)
    return weights / weights.sum()               # normalise to sum = 1


def distribute_flows(alpha: float, total_flows: int, tiers: list, seed: int = None):
    """
    Assign `total_flows` flows to tiers according to the Pareto distribution.

    Returns
    -------
    counts     : int array  – number of flows per tier
    bytes_arr  : float array – total bytes per tier
    flow_pct   : float array – % of flows per tier
    byte_pct   : float array – % of bytes per tier
    """
    rng    = np.random.default_rng(seed)
    probs  = pareto_tier_weights(alpha, tiers)

    # Multinomial draw so counts sum exactly to total_flows
    counts    = rng.multinomial(total_flows, probs)
    bytes_arr = counts * np.array([s for _, s in tiers], dtype=float)

    total_bytes = bytes_arr.sum()
    flow_pct    = counts    / total_flows  * 100
    byte_pct    = bytes_arr / total_bytes  * 100

    return counts, bytes_arr, flow_pct, byte_pct


# ── Main ───────────────────────────────────────────────────────────────────────

def main():
    counts, bytes_arr, flow_pct, byte_pct = distribute_flows(
        ALPHA, TOTAL_FLOWS, TIERS, SEED
    )

    labels     = [label for label, _ in TIERS]
    total_bytes = bytes_arr.sum()

    # ── Console table ──────────────────────────────────────────────────────────
    header = f"{'Tier':>8}  {'Flows':>9}  {'Flow %':>7}  {'Bytes':>12}  {'Byte %':>7}"
    print(f"\nPareto Flow Size Distribution  (α={ALPHA}, {TOTAL_FLOWS:,} flows)")
    print("=" * len(header))
    print(header)
    print("-" * len(header))

    for i, (label, _) in enumerate(TIERS):
        print(f"{label:>8}  {counts[i]:>9,}  {flow_pct[i]:>6.1f}%"
              f"  {fmt_bytes(bytes_arr[i]):>12}  {byte_pct[i]:>6.1f}%")

    print("-" * len(header))
    elephant_flows = counts[3:].sum()
    elephant_bytes = bytes_arr[3:].sum()
    print(f"\n  Total bytes : {fmt_bytes(total_bytes)}")
    print(f"  Elephant flows (≥1 MB) : {elephant_flows:,}  ({elephant_flows/TOTAL_FLOWS*100:.1f}% of flows)")
    print(f"  Bytes from elephants   : {fmt_bytes(elephant_bytes)}  ({elephant_bytes/total_bytes*100:.1f}% of bytes)\n")

    # ── Plot ───────────────────────────────────────────────────────────────────
    x      = np.arange(len(labels))
    width  = 0.38
    blue   = "#378ADD"
    coral  = "#D85A30"

    fig, ax1 = plt.subplots(figsize=(11, 5))
    ax2 = ax1.twinx()

    bars1 = ax1.bar(x - width / 2, flow_pct,  width, label="Flow count %", color=blue,  alpha=0.85)
    bars2 = ax2.bar(x + width / 2, byte_pct,  width, label="Bytes %",      color=coral, alpha=0.85)

    # Labels on bars
    for bar, val in zip(bars1, flow_pct):
        if val >= 0.5:
            ax1.text(bar.get_x() + bar.get_width() / 2, bar.get_height() + 0.3,
                     f"{val:.1f}%", ha="center", va="bottom", fontsize=8, color=blue)
    for bar, val in zip(bars2, byte_pct):
        if val >= 0.5:
            ax2.text(bar.get_x() + bar.get_width() / 2, bar.get_height() + 0.3,
                     f"{val:.1f}%", ha="center", va="bottom", fontsize=8, color=coral)

    ax1.set_xticks(x)
    ax1.set_xticklabels(labels)
    ax1.set_ylabel("Flow count (%)", color=blue)
    ax2.set_ylabel("Bytes carried (%)", color=coral)
    ax1.tick_params(axis="y", labelcolor=blue)
    ax2.tick_params(axis="y", labelcolor=coral)
    ax1.yaxis.set_major_formatter(ticker.FormatStrFormatter("%.0f%%"))
    ax2.yaxis.set_major_formatter(ticker.FormatStrFormatter("%.0f%%"))

    ax1.set_title(f"Pareto Flow Size Distribution  (α={ALPHA},  {TOTAL_FLOWS:,} flows)",
                  fontsize=13, pad=12)
    ax1.set_xlabel("Flow size tier")

    lines = [bars1, bars2]
    labs  = ["Flow count %", "Bytes %"]
    ax1.legend(lines, labs, loc="upper right")

    ax1.set_axisbelow(True)
    ax1.yaxis.grid(True, linestyle="--", alpha=0.4)
    fig.tight_layout()
    plt.savefig("pareto_flow_distribution.png", dpi=150, bbox_inches="tight")
    plt.show()
    print("Plot saved to pareto_flow_distribution.png")


In [ ]:
main()